# Sierra periods

Reads the Sierra snapshot in `data/sierra-raw-works.parquet`, pulls the same period subfields as
`period_extract` into `data/sierra-periods.jsonl`, pairs each production date with the record's 008
range in `data/sierra-008.jsonl`, and diffs the distinct strings and shapes against `data/periods.jsonl`.

In [1]:
import json
import re
from collections import Counter
from pathlib import Path

import pyarrow.parquet as pq

from adapters.transformers.ebsco.parsers.field008 import Field008
from adapters.transformers.marc.parsers.period import parse

PERIOD_SUBFIELDS = {"648": ("subject", "ay"), "650": ("subject", "y"), "651": ("subject", "y"), "655": ("genre", "y"), "260": ("production", "cg"), "264": ("production", "c")}

PERIODS_FILE, F008_FILE = Path("data/sierra-periods.jsonl"), Path("data/sierra-008.jsonl")

if PERIODS_FILE.exists() and F008_FILE.exists():
    periods = Counter({(r["tag"], r["path"], r["text"]): r["n"] for r in map(json.loads, PERIODS_FILE.open(encoding="utf-8"))})
    with_008 = Counter({(r["text"], r["range_008"]): r["n"] for r in map(json.loads, F008_FILE.open(encoding="utf-8"))})
    print(f"reused {PERIODS_FILE} and {F008_FILE}")
else:
    periods, with_008 = Counter(), Counter()
    records = deleted = 0
    for batch in pq.ParquetFile("data/sierra-raw-works.parquet").iter_batches(batch_size=20000, columns=["body"]):
        for body in batch.column("body").to_pylist():
            bib = json.loads(body).get("maybeBibRecord")
            if not bib:
                continue
            data = json.loads(bib["data"])
            records += 1
            if data.get("deleted"):
                deleted += 1
                continue
            fields = data.get("varFields", [])
            f008 = next((f["content"] for f in fields if f.get("marcTag") == "008"), None)
            try:
                range_008 = Field008(f008).maximal_date_range if f008 and len(f008) >= 15 else None
            except NotImplementedError:  # a blank or unknown date type
                range_008 = None
            for field in fields:
                if (spec := PERIOD_SUBFIELDS.get(field.get("marcTag"))) is None:
                    continue
                path, codes = spec
                for sub in field.get("subfields", []):
                    if sub["tag"] in codes and (text := sub["content"].strip()):
                        periods[(field["marcTag"], path, text)] += 1
                        if path == "production":
                            with_008[(text, range_008)] += 1
    print(f"{records} bibs, {deleted} deleted, {sum(periods.values())} period strings, {len({t for _, _, t in periods})} distinct")

    with PERIODS_FILE.open("w", encoding="utf-8") as f:
        for (tag, path, text), n in periods.items():
            f.write(json.dumps({"store": "sierra", "tag": tag, "path": path, "source": "marc", "text": text, "n": n}, ensure_ascii=False) + "\n")
    with F008_FILE.open("w", encoding="utf-8") as f:
        for (text, range_008), n in with_008.items():
            f.write(json.dumps({"text": text, "range_008": range_008, "n": n}, ensure_ascii=False) + "\n")
print(f"{sum(n for (_, r), n in with_008.items() if r)} of {sum(with_008.values())} production strings have an 008 range")

reused data/sierra-periods.jsonl and data/sierra-008.jsonl
1245534 of 1270263 production strings have an 008 range


In [2]:
existing = Counter()
for line in Path("data/periods.jsonl").open(encoding="utf-8"):
    row = json.loads(line)
    existing[row["text"]] += row["n"]

new = Counter()
for (tag, path, text), n in periods.items():
    if text not in existing:
        new[(path, text)] += n
print(f"{len({t for _, t in new})} distinct Sierra strings not seen before, {sum(new.values())} occurrences, out of {len({t for _, _, t in periods})} distinct")
for (path, text), n in new.most_common(40):
    span = parse(text)
    print(f"{n:6}  {path:10} {text!r:50} {span and f'{span[0]} .. {span[1]}' or 'no range'}")

18729 distinct Sierra strings not seen before, 39374 occurrences, out of 88375 distinct
   685  production '1887)'                                            1887-01-01 .. 1887-12-31
   583  production '[1890?/1910]'                                     1890-01-01 .. 1910-12-31
   421  production '1980?'                                            1980-01-01 .. 1980-12-31
   310  production '1930s?'                                           1930-01-01 .. 1939-12-31
   269  production '1930s-1980s?'                                     1930-01-01 .. 1989-12-31
   232  production '1940s?'                                           1940-01-01 .. 1949-12-31
   224  production '1926-c.1952?'                                     1926-01-01 .. 1961-12-31
   203  production '1920s-1930s?'                                     1920-01-01 .. 1939-12-31
   171  production '1950s-1960s?'                                     1950-01-01 .. 1969-12-31
   171  production '1985?'                               

In [3]:
def shape(text):
    return re.sub(r"[a-z]+", "a", re.sub(r"\d", "9", text.lower()))


existing_shapes = {shape(t) for t in existing}
new_shapes = Counter()
for (path, text), n in new.items():
    if shape(text) not in existing_shapes:
        new_shapes[(shape(text), text)] += n
print(f"{len({s for s, _ in new_shapes})} new shapes, {sum(new_shapes.values())} occurrences")
for (s, text), n in new_shapes.most_common(40):
    span = parse(text)
    print(f"{n:6}  {text!r:50} {span and f'{span[0]} .. {span[1]}' or 'no range'}")

1142 new shapes, 6973 occurrences
   583  '[1890?/1910]'                                     1890-01-01 .. 1910-12-31
   139  '[1920/1940?]'                                     1920-01-01 .. 1940-12-31
   101  '1945-c1996?'                                      1945-01-01 .. 1996-12-31
    96  '197?/198?'                                        1970-01-01 .. 1989-12-31
    69  '[1890/1910?]'                                     1890-01-01 .. 1910-12-31
    52  '[1850/1880?]'                                     1850-01-01 .. 1880-12-31
    42  'Aug. 67 [August 1967]'                            1967-01-01 .. 1967-12-31
    26  'Feb.y 1st 1777.'                                  1777-01-01 .. 1777-12-31
    24  '1918-c.1950s ?'                                   1918-01-01 .. 1969-12-31
    24  '18th century - c. 19th century?'                  1700-01-01 .. 1909-12-31
    23  '[1860?/1900?]'                                    1860-01-01 .. 1900-12-31
    23  "1930's-1955?"                    

In [4]:
unparsed = Counter({key: n for key, n in new.items() if parse(key[1]) is None})
print(f"{sum(unparsed.values())} of {sum(new.values())} new occurrences give no range")
for (path, text), n in unparsed.most_common(30):
    print(f"{n:6}  {path:10} {text!r}")

1061 of 39374 new occurrences give no range
   138  production 'No date.'
    81  production '[s.d]'
    60  production 'Nd.'
    31  production 'Date unknown.'
    30  production 'early 17th C'
    17  production '[Date of creation]'
    15  production '[s.n.]'
    13  production 'Not dated'
    13  production '[Date of Creation]'
    11  production '[1898], ©1898.'
     9  production '1946-1951, 1953-1955'
     8  production 'n.d.].'
     8  production '18thC?'
     6  production '?'
     6  production '1933 or 1953.'
     6  production '1950 (or 1942)'
     5  production '197u.'
     5  production '[1831 or 1832]'
     4  production '[18-]'
     4  production "['94?]"
     4  production '1878-1879, 1879-1880'
     4  production 'June 67.'
     3  production '[1865 or 1866]'
     3  production 'cno date.'
     3  production "Erscheinungsdarum, Februar '88"
     3  production '19?'
     3  production '1846-1848, 1862-1874'
     3  production 'undated,  20th century'
     3  production